
# 예금보험공사 RAG 챗봇 데이터 수집 파이프라인

이 노트북은 예금보험공사·금융안심포털의 **42개 URL Manifest**를 기준으로 다음 작업을 수행합니다.

1. Manifest 검증 및 실행 대상 선택
2. 정적 HTML 수집
3. 이미지·첨부파일·링크 메타데이터 추출
4. 동적 조회 페이지의 초기 HTML·폼·스크립트 구조 진단
5. 결정론적 HTML → Markdown/JSON 변환
6. 응답 메타데이터·해시·실행 로그 저장
7. 페이지 문서와 기본 청크 JSONL 생성
8. 결과 전체 ZIP 다운로드

## 수집 정책

- `include_full`: 공개 본문 전체 수집
- `include_partial`: 공개 페이지의 초기 구조만 진단하며, 검색 조건을 임의 생성하지 않음
- `link_only`: 로그인·본인인증·신청·자가진단 결과를 수집하지 않고 서비스 설명과 공식 URL만 저장
- `review`: 사람의 최종 결정 전에는 실행하지 않음
- `exclude`: 요청 자체를 보내지 않음

> 이 노트북은 로그인 우회, 본인인증 자동화, 개인정보 조회, 신청 제출을 수행하지 않습니다.  
> 기본 요청 간격과 낮은 동시성으로 실행되며, `robots.txt` 정책을 확인합니다.


In [ ]:

# Colab 런타임에 필요한 라이브러리 설치
%pip -q install "requests>=2.32,<3" "beautifulsoup4>=4.12,<5" "lxml>=5,<7" "pandas>=2.2,<4" "tqdm>=4.66,<5"



## 1. 실행 환경과 설정

기본 설정은 정적 페이지·자산 페이지·링크 전용 레코드를 처리합니다.

- 특정 업무만 실행하려면 `SELECT_BUSINESS_FUNCTIONS`에 업무명을 입력합니다.
- 테스트 실행은 `MAX_URLS = 3`처럼 제한합니다.
- 동적 공개 페이지를 브라우저로 렌더링한 초기 화면까지 저장하려면 `ENABLE_PLAYWRIGHT_SNAPSHOT = True`로 바꿉니다.
- Playwright도 검색 폼 제출이나 로그인은 수행하지 않습니다.


In [ ]:

from __future__ import annotations

import hashlib
import json
import mimetypes
import os
import re
import shutil
import subprocess
import sys
import time
from dataclasses import dataclass, asdict
from datetime import datetime, timezone, timedelta
from pathlib import Path
from typing import Any, Iterable
from urllib.parse import urljoin, urlparse
from urllib.robotparser import RobotFileParser

import pandas as pd
import requests
from bs4 import BeautifulSoup, Tag
from IPython.display import display
from requests.adapters import HTTPAdapter
from tqdm.auto import tqdm
from urllib3.util.retry import Retry

KST = timezone(timedelta(hours=9))

# -------------------------
# 사용자 실행 설정
# -------------------------
SELECT_BUSINESS_FUNCTIONS: list[str] = []
# 예: ["예금자보호제도", "예금보험금 안내"]

RUN_DECISIONS = {"include_full", "include_partial", "link_only"}
RUN_WAVES = {"W1_STATIC", "W1_ASSET", "W2_DYNAMIC", "META"}

MAX_URLS: int | None = None
# 최초 테스트 권장: 3
# 전체 실행: None

REQUEST_DELAY_SECONDS = 1.2
REQUEST_TIMEOUT_SECONDS = 25
MAX_RETRIES = 3
MAX_ASSET_BYTES = 25 * 1024 * 1024

RESPECT_ROBOTS_TXT = True
DOWNLOAD_ATTACHMENTS = True
DOWNLOAD_IMAGES = True
DOWNLOAD_VIDEOS = False

ENABLE_PLAYWRIGHT_SNAPSHOT = False
PLAYWRIGHT_WAIT_MS = 1500

CREATE_BASELINE_CHUNKS = True
CHUNK_MAX_CHARS = 1200
CHUNK_OVERLAP_CHARS = 150

USER_AGENT = (
    "KDIC-RAG-Academic-Crawler/0.1 "
    "(low-rate public-document collection; no authentication automation)"
)

RUN_ID = datetime.now(KST).strftime("%Y%m%d_%H%M%S")
RUNTIME_ROOT = Path("/content") if Path("/content").exists() else Path.cwd()
OUTPUT_ROOT = RUNTIME_ROOT / f"kdic_crawl_output_{RUN_ID}"

PATHS = {
    "raw_html": OUTPUT_ROOT / "raw" / "html",
    "raw_assets": OUTPUT_ROOT / "raw" / "assets",
    "response_meta": OUTPUT_ROOT / "raw" / "response_meta",
    "parsed_markdown": OUTPUT_ROOT / "parsed" / "markdown",
    "parsed_json": OUTPUT_ROOT / "parsed" / "json",
    "dynamic_diagnostics": OUTPUT_ROOT / "diagnostics" / "dynamic",
    "screenshots": OUTPUT_ROOT / "diagnostics" / "screenshots",
    "logs": OUTPUT_ROOT / "logs",
    "processed": OUTPUT_ROOT / "processed",
    "manifest": OUTPUT_ROOT / "manifest",
}

for path in PATHS.values():
    path.mkdir(parents=True, exist_ok=True)

print("실행 ID:", RUN_ID)
print("결과 경로:", OUTPUT_ROOT)


## 2. 42개 URL Manifest 불러오기

In [ ]:

# 현재 프로젝트의 42개 Manifest를 노트북 내부에 압축하여 포함했습니다.
# 별도 CSV 업로드 없이 기본 Manifest를 자동 생성합니다.

import base64
import gzip

EMBEDDED_MANIFEST_GZIP_B64 = """H4sIAHmDVGoC/+1dbW8TWbL+PtL8h1akXSXaXhzbCS/3w1xlgJmJBmaZwOzofrIcuxO8OG3T3Q6T/TAy0EGGZJawNwYH7FznEkiCgtYJDnF2g66Un+Pu1v6FW3XqdLvbbjs2hLCajYRiu19On656TtVTdeoc/vmP/0to0lRETokZJRlJxMXxjJqQJVWNTGTkmJZIyaIWVSYlLdJ6Ih2dlCJaQktKoprKKDEpAm2IKjQYkaNTkphSEpMJOZqMpJUEfNdmRDmlTEWTiT9L8UhciiVUbMX+ElGkqOq0OpOWxAlJi12PqJoS1aTJGTihqJICjaUmEvDIaEa7DvfczCQUKS5GVRX6mE4lE7EZMZaSNUnWImosBc0o0nRCutW4NKZEbyUjt6LTEv+qalEto4rST2kppkHXUhktndFEWYLvvFNOi5mpqaiCL6JJ6uefBcULV34/OBgUzUKuXsuZywvGm6pVqJnlovFQb3OUflh59uO6pqXV/wgEbt26depGPBE7lVJO3VACajoQT8Obw8tqgSvw5+qMquHnpSlNDahSEnp6NabIp+Ip/hRs9Umh/mbHvLMpWnndzBXEhBxLZuISaCyZFM0ns8ZmVTDXcsZmzSwV4Ntts/xIqO/u17cqxkpRgDvNpxtCvZI1HryA/gnY2lxJMPM5405VsP5SMEtVcy0rosASsQhqSkSxSqqmRq5rU0kRXyHCT08HQUzwTwZNQgci46k4Ck78MRi5em3k2uh5lGIE5AgyF81nIKSa8M21y5eE3wn1nU1zpSLAO5krVfioQZfhsLGxaN3NGr9UoB+WXsE3N+8VQ0PCD2OXxKBZqhlze0JDHLaQhaD5eAceUH+zL5jVHHygVvIrgrGn03UCnDX3lgRTXzVezYqffxYixYZ6Uqz/0V40fKLZZs3WaxXoclAw5rPWo6KVf2282kA91itFY3ZBMEv75l7BR1DwqgL9Np7PGy9A8eXbxst3Vh4e+mjBfJIz5lbxGlRYccd6Og/34EPMuTJHkFnSQZDZeu2+lS8YDxYRI8bLzQZGwoSR8HsMfnzQ3dvCF97Hf4FvZT5d7AoyhBS3abiamVJOUOPYA7cdELjAPeI2/3fWeFHoN0tZ6/H9g10SGJw42LVym6i3tVX4B2KC74Lx31sDgnGnYD6pguqHSPVDH656+G79Nfehqr+mTMJnPHaCgW4xQHJHS4JOgM7QsX7SPhgCC46h3lEw8HTjrY7myFzNNmzAMAFhuCcg9NXflOuVkqnvQ4dB9V+NfA9/jb/VQD9GeQ/usO7tWPkN6IK5/bofVQDv9gDe4K0OiMWuwtPNvdeiYM6tQqMAzCpoCZDs8X4DfQ6cJhKy6sZTbCowPm4Tia+iN79TolPaSDo5g+DhEsrnzLlN6+Gmpdc+NngmojebQIMdjsDhT+Br+t3cgeyCPWwHWl1PXyvfEOE9dTAU4HOy5ts8GRhUnnmnQlcLdDnor3AbDUw5D1IQrGe643tq5luQXzlnVtbRe4G3Mp5vIlL6AHWnCXWnP5ydYIs9MRTAyuEkBaBirj06RrQwE/Np0MJI5fN9FCdwhHplCXporM0LocHQsDGrC5yCom15/ALNiWC3YD4BED25Z/wCstgpMuJxZ9NYyTr6rol2I+fMZ4t+xNbdKjWBtsTdTNUszQNizhBiznwIYuiZ/UChzKXaQDvUxBOB8T+nx5Oy46fGY8oV/G2HL3+YVtQ2wJF+YohxPulZaKLrOwvYEeBygJTTMARBsIAO8OIMVy5AGXM7xpssxxVSPDOvG3MFQMyEpEgyRKmx1BTEkgmM6+wHqTcSaQc5/I949dvRK+xMGgLB8Rk7uiR0AynEMWvmH6CEfXTq6btAnRfByNd3Xh/smn9fADIBWoavpX0AG3wWbrNTxsNZNC34Qi0KZzplDeL4AEIMqj0rfjnqikG5PMHG0Ljp7TD4IfI65lyR+bw93dSL4HS6MhIX0urU6Dj3JFdSqnohNiWfBDUtpiN0KmiLmQlXaBfXQFPgCgAT81kIPegr0AR9C4wCZ6V2QyUIYotlq5A33ugezdkK//yzc4SU0IcihT2QOycEjO95Ov0esFFSsU+OGOhWDFNd7w+ZkatXL147SsSEmuTeHjI8jAGSKRgQVNwtGW+zSDQpmIVXhjNgblhr8ElH9+bhCxMaNe8w3OAgoSb8YaiBFzKWs//ZGx4u/pROyifmw595hHrJidRy/fCiXNe13MEuhpvARu0DA6T6PPtwVB8k1Q8dlWtxHWIP9onK3yMA/hKQ8nViWsIg+CtZ1dqgBEYBUnDmvIX623lz+YUDGaADWiIKqFmpQDcE+KhvVQ92HQwY+ip+mQXhPF1k1lZnfF4vIXCMv64ieJCU1LcroAnkf/Q4gVPh+IwcnQKUJFOpG5m08xNfNqJdl+RIOhmduaUkJq9rhCrvDW50AQCVmYgiqZmkpoozkir+GIpc+K/vRi57McZ7ZGwvmkVdIMQFqH8+4OprRpe5lPXjtaJgzYOeKm6dMY+zDaq8z2/mgWgzI6IotU/0b6EFNW6ImE8g/M46T2FjFhEaEn/4jpEfCqk9QTQ3mJ1OeY7xoJtBIMCN7RcC3e2+sKd4iQXWo/JNdG3s+9eZ+IlJa2PSwi0mDeH3aoOZK7ePc2mjYbgYYks1aNfQd4EoVTbwoVyR+Gs73+BCwTAhJ3QEyHFna5AP3ZtH64q9YYc8j3b4EqNvHNxl3VpatJVKPWyXuMlEA1GA0iSi6AKiDa3eNSn5fUb1g9bxp3LaQ4uldI4fWmGgTl4suOBVLMOL4euQDgT79bd0Q18QMcsGvyCcrG+/EzAk4udJpSx76MZGvZo1yuu8LdAshN9cQJ5gPDhE2At/Aux1jy6yXNqkgqZrVFPil7Xr8ROIdWu9ACMPAXvBg92ww925NtpSd4+u2ewE02Rh3cCEAMsRrFqLOv1qtmwOaxsmdA19PJ8I0Dbv/YJ9IOWjApcLdFVP3vEbJS19k1DRQXaVVfzVO8RmGhZuk14UXTqwvQhx164YGFKutRK4oH5rcd9paUCwDZ7Hv+qNZ0Ev+XPsOVG9iImGB6sIvNPi5WuUiapsAFLNeyss1V0pWEsFG129ncKY9NkCT1UD7lxXuC8Q6AoIMDug78aUorI/aMu6ZWD9mKDH+RZbQwNeCJIimbTtrCDG3XQXHv5h7BIMW85iaIw2Kx1iioqpl8XpRFxK+UAwwmpbVB8k0h1TkhaNR7WoiHCEQCKj0h2NEx8lJUFAGzosCm2nVK4zNmx31/EYmDqaBHXZsjMEqdCngNQh+SsPnk6yVn4QGToVFHyE6nKA0AA0ygoxtirM9DxcAuSI3F9S4gpjxFwBGT02RakqYdiYW69j2NicrzpLkAl/Asi4nHy3wEFyhcTqJCRsBU+XtOmQmI/CyEdon7ddyfDgOcLJ0HvixHcOvx00+D3dzsZfVuPK19PjNzpMyLfMlBV0c3kTnTOWtlCaBxw/+GcnA1SGuKCGmICzEDEbC+tsJvWop8Bw1B/sovaaFcbYA9MZsoeNClYvMPjqNRjpLBB7jfioV/IikRqq04Pwv1apV3TmSOEdENPFMk1+tHUvoBBP0BUaJH0PH6W+r/3hihAc7Fqv11Lp4OC/c4XFId5W6J1F2CH3TtGYW7XTt0xeoJj6VpUlbPPvEABBAsDpI3AMni54hz0ODeAzwMu/YOmFUoGA3RYkCSUQjccVVQqMsI8RTZMva5pyEmZ3Qk9DzNhbEHK5CEbiYBcJQu1+S6KnuWZnecFc07l+4T4Qg+hpiPsOOrhdxaflsJJUoPYd1sFkhiU5oRCh68xHQJeT06FOOckc9yxvI8PjJ6SO4LMTPuh1CILO1H0v6Z5+c3sdCdv8PEi5KUzqrC1nZlMvAhAx4LTy6/V/FOzJk6imRWPXp7D+3i86cs6qBEzX5QTOeOqWnExF455LXcGS6+hHocEdEOqZqG9PcBr3w6VYMuIBpbtF5vDRMTJZotOH4Y34DBM+z/5L4bMHZKaVo0elT+UEhLIs58bu+vUi0quEZp5dQL7YHTRbRdjvV7QCqMBMFRZRu9tFhgYSZ24JGOnLTUTqECH13DEi1RvneQL/jrjUFBV5+tW0ovlkAI7fZXfIAHTps48YaJ1Jnu1Fc0WAA4/rPcCyK1eWgcOtdAj/2QKPvy8YL1+zyx+sWvfKgCZrdtEAWRb3mRCdtT3DDGDA24/RFDKaCkeM8h6bGajvFgHxXaNsTEpeit5KjU1mkoeDrI+bPnocSMC6vSlyOGEENVdkc1nm20VWHlnWITAGaf3Wjup04MwDfX6JTpay4S/BmmWCd2c+6bCdk0ZA6yxzDYYBsyf17SyoRrBuL2BWGuyBsVISk9JkNIkVDbSIMCHHpZ+aTGoyId/gxpSuZhdxIONJXFGYimWYzYzAnRnJZU/xgk+QDeWlqCzspSGNyftXG2wEU9xCwoQAl8EBmzfuV5sLb+0kRjvc0c1Y0lW0Ht+HaJotZYPXwVSIQwAoOx8MHiPqsTx5acPUy9bdUtdQv6pltPPXY93DnNIHNvBWivVd9D8EY9HG6do8t5bNwKY+YjUP0n4yEi35fKdVgZq1faQz/cEg7jwJwYC4jsrxhDwZUVIZTVLaI5rOczDD9/FURo7TBX7J/WNAcxM/AGj5hl5caNQ6z9+3idrh8kCb+I1AguZgsWKU9vkgYMkb1r491bSmm8uzWCixgfm7EE0NBEPHShMwvciTiZ1n0ptR/UdVG4ulT0o02qR6h9qWaBCkAIHP90EvzlS5SxOcAzDa/rSKU29zRZYtRCKK6/z4nXbmN0QzBMHwR04E9ZQCmmIhDo90TlJA7xm4IGjYG9NhRkK8WeNWK9Xn0aFf3pkXomH2nJ0VcQ0GCNB4wdYKMy7VSATRvEJw6DjQ5TP9hB6rVnHWcVCPefmktZQ/JOTmMCTc4fTD98mJ2Ghc1mbG1KTWOdJu8bSMmEVScnJGBC7D6o91oO12kSwaeAgR/+c+uICSqW81phPQ5XJIEq/ivpUq/LCEQUBFl2+zhxHDScjgQ6MxLTEtRaD7E5HYdSl2Q/Q4S+qL91Bnr+tzO1x9+eK1EQ+mHRrwahbQ6+ki+10zX5bsGtbf2a+GC1Qb9R5ElRkhB74I5wCeIKm7dvTCCSRcDv4SF/8wR4hl9RsVex0oXMhWhfnPmhHe+8QztHqKxgpcvV0F0bKF8T6TmZ7yW7cWcQE8TakEhz/ZVCsNxx6mWrFasotKIzbYP3BuzW07j2qOrbk8aKhtlTYa1u0q4o9smb2alZLdh5UH8eQk8w+eBvqHofWf+X4aA406Q2q2aYLO8bjhoHhhhCqDtnRcrbeCQnKWCxx6bHkTM5+OGXSd67zHhjIeSI4nteifprTAJfgyAl8wV+N872rLDXonoT9sbOf4wwV6Oo6qsz+fG8SaPByjMO5xOcObOzDOWETA1gAMOOr/duy35z3db7haxjbBGvJVA6uPjDc7xkLBfFzlISU3EBx3HKEfA2NNEBs+rMDHoylKB4KZ2MaibXf+14EKSCXHzFwzSEIEktBRggSG4hu7ZE345/49gapym+bje4DNqDyRGpVvXlBjqowGpeu6jYYvpPwoOQW09RRSsiJk1k8YRbx02HYraOX/lmXmhdwhhalMGeRLXL63nMWVLbg7kiQDOKK4o5EqKdOJmNSVN8QVJQw2/KZIXFJjSiKNWz59BA/owG0Y6z1odBFPEVo8GBUHsSU3rGVaJLtu18Pyom23eMl5ti44Odh1jUXeL5bLclqjPti64IABlIYJpeH3QmmHRdOKlMSdswK4XPpbRY4B2KIAxJOyoKYIYBirS4ZPhfEPc38ECqfERBeYZp03YBsB/bLIalj1mrWcswljn51idlkj0bvmGRpdQJJVXO/HAoP5rHBmEMz9bwZEATqHwUE1a6/HflRslJtgHBAeIqAMHY05g74+3QDsglsQ6Giv1uu8IsXHYtPKDP44HFf28jaWRaMt1ER3L4D2QCdYuUXZXprO1uU3FuTzlfjEkZBmYLyQX7fyZXQ3xJS49ROj8T9FY2w+jTxTWuK5M3oy38ONH3XA1nQVmq6xi38cvfijc8bZZs6aX2CxBxsCPENHO27QikBXiATjtJloDbclWv4ysQqPzPw7bpK6YFstLbCZMSy+L/rpnsvc5TyHCW3DR4I2speI/K3ZXmF2RZW/VN4PXphqw3R2pYj5W9aHg13qBQtUWt/iV44tEgibfiYouEVjT309WOwEro5N2NL1lrMgnE4TnE4fCZz4I8HxuqYBuofT2ORRwIk9/QRGbgwwkXwYjNxN9Id/ZpvVgC5QfpjWLwy4MHWGMHWme0xRMgCiiVoO58te4hZ5WCm6pQO3w3maZwsBzwokRvc7k3s1Df8UbSIjxwMIsK/k+AVpXLuQlCeRz3e3PJdPNjUINCW5qFOYFqgUG1ViLOXI6RUPrLffYaoG+SZ7Uz7D0cR53Gw+nU4ilUcK3guX75DZOh5233covW+7wMkWmb9QD8FrK+O3x/3yAqMmXhRhXqQZRs7zHOp/lgB8tmsA+9eIuyLTpiVzh9UPuwLQNqlYX9bmiZCag2K7TvrhrPHynW17+Lw9K8u5Ddac20NuoHjzWFL8L2MDO8HLDhazHt7WkAqvGu6Gq1nFKqZDHlYEO353LGJnMfOZxJLuHfK4/Rog65z4Dd9HuJQ15u6DxJmDnysDgvyPtTONc6vIlpZ1nqy0t5mkU1R0gCvP3W1SkxRkwBeCaw/m87wcm7oQSyonOxu0Lww53bqzwZKO68QxHqVEGoxgmnkSUCflPG01aQee4UGec6VMfhWeB/dhKaiz+MV8WrH+UnMvggFtNu97MDRIUAt1DTV/I9aCn66NGMLlipLWEDL/zmshHH2D5lj9FTk20N+ebsA7QDvLL9oWXrrQwvfwQbTY1as0on0bpJ2uWbkFIBKkRCZoKEi4CLfBBZtrZ0Fnv/E2B5DCSiOgzOz8wAfbo9Y2m+wRL8rowSyNTqsT0eQPspIcTU4muzdQfbj3IlKvVjmwdBOcfVFzVq/aiVkcnWvsW1+fz51wECWuP0c3aqwtoFDZlZ3l2teHLfLaOetusb432+e49Y79czfWwIO3z7xdTN6gNanxJzQ22/GNfmz++cldf4sJosLtah1eC9kh39ujYNwpiIL1dIUV2hmVCqCRlcjz1+d2kubJcV8n3HqT+CXTDhdbn+jdKtwLYDRIfo/w1QIWreCQC9GQG+raFPsMKHNNh+atgo5FbJxr9FNwZhMQJj2bs2OytNB2887pVCyAGzTACAmcjyqSpIxJkwlVG2H/4UMgCV99wyG2XwAWxtuTUJ4e0ENZ2atfd/lMNqXqx0a+tm0nV7utmaq9kWcsJatO5f3Rzay2ypvHaVQysoTM0E8BuIPva75NGE8IMpfJNx6dFzpoyt6soYp1b5y30njrO0zZjEraWzGw0sWi3gjMPbqnBSOO9gWnhu7/AYEZ+Ef+YwAA"""

MANIFEST_PATH = RUNTIME_ROOT / "kdic_crawl_manifest_42.csv"
MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
MANIFEST_PATH.write_bytes(
    gzip.decompress(base64.b64decode(EMBEDDED_MANIFEST_GZIP_B64))
)

print("Manifest 생성 완료:", MANIFEST_PATH)
print("크기:", MANIFEST_PATH.stat().st_size, "bytes")


In [ ]:

# 필요하면 사용자 수정본 CSV로 교체할 수 있습니다.
USE_CUSTOM_MANIFEST_UPLOAD = False

if USE_CUSTOM_MANIFEST_UPLOAD:
    from google.colab import files

    uploaded = files.upload()
    csv_names = [name for name in uploaded if name.lower().endswith(".csv")]
    if len(csv_names) != 1:
        raise ValueError("CSV 파일을 정확히 1개 업로드해야 합니다.")

    MANIFEST_PATH = Path("/content") / csv_names[0]
    MANIFEST_PATH.write_bytes(uploaded[csv_names[0]])
    print("사용자 Manifest로 교체:", MANIFEST_PATH)


In [ ]:

REQUIRED_COLUMNS = {
    "url_id",
    "business_function",
    "page_title",
    "source_url",
    "site_name",
    "normalized_decision",
    "decision_reason",
    "page_type",
    "fetch_strategy",
    "parser_profile",
    "auth_required",
    "asset_policy",
    "content_scope",
    "crawl_wave",
}

manifest_df = pd.read_csv(MANIFEST_PATH, encoding="utf-8-sig", dtype=str).fillna("")

missing_columns = sorted(REQUIRED_COLUMNS - set(manifest_df.columns))
if missing_columns:
    raise ValueError(f"Manifest 필수 열이 없습니다: {missing_columns}")

if manifest_df["url_id"].duplicated().any():
    duplicates = manifest_df.loc[
        manifest_df["url_id"].duplicated(keep=False), ["url_id", "source_url"]
    ]
    raise ValueError(f"url_id 중복:\n{duplicates}")

if manifest_df["source_url"].duplicated().any():
    duplicates = manifest_df.loc[
        manifest_df["source_url"].duplicated(keep=False), ["url_id", "source_url"]
    ]
    raise ValueError(f"source_url 중복:\n{duplicates}")

invalid_urls = manifest_df[
    ~manifest_df["source_url"].str.match(r"^https?://", na=False)
]
if not invalid_urls.empty:
    raise ValueError(
        "HTTP/HTTPS가 아닌 URL이 있습니다:\n"
        + invalid_urls[["url_id", "source_url"]].to_string(index=False)
    )

# 실행 대상 필터
target_df = manifest_df.copy()

if SELECT_BUSINESS_FUNCTIONS:
    target_df = target_df[
        target_df["business_function"].isin(SELECT_BUSINESS_FUNCTIONS)
    ]

target_df = target_df[
    target_df["normalized_decision"].isin(RUN_DECISIONS)
    & target_df["crawl_wave"].isin(RUN_WAVES)
].copy()

if MAX_URLS is not None:
    target_df = target_df.head(MAX_URLS).copy()

print("전체 Manifest:", len(manifest_df))
print("이번 실행 대상:", len(target_df))

display(
    manifest_df.groupby(
        ["business_function", "normalized_decision"],
        dropna=False,
    ).size().rename("count").reset_index()
)

display(
    target_df[
        [
            "url_id",
            "business_function",
            "page_title",
            "normalized_decision",
            "crawl_wave",
            "fetch_strategy",
        ]
    ].head(20)
)


## 3. 공통 유틸리티

In [ ]:

ATTACHMENT_EXTENSIONS = {
    ".pdf", ".hwp", ".hwpx", ".doc", ".docx",
    ".xls", ".xlsx", ".ppt", ".pptx", ".zip",
    ".csv", ".txt",
}

IMAGE_EXTENSIONS = {
    ".png", ".jpg", ".jpeg", ".gif", ".webp", ".svg", ".bmp",
}

VIDEO_EXTENSIONS = {
    ".mp4", ".webm", ".mov", ".avi", ".mkv",
}

NOISE_SELECTORS = [
    "script", "style", "noscript", "template",
    "header", "footer", "nav", "aside",
    ".skip", ".skip-navigation", ".skipnav",
    ".gnb", ".lnb", ".snb", ".breadcrumb-wrap",
    ".footer", ".header", ".quick-menu", ".quick",
    ".util", ".utility", ".pagination",
    "[aria-hidden='true']",
]

MAIN_SELECTORS_BY_SITE = {
    "예금보험공사": [
        "#contents", "#content", ".contents", ".content",
        ".sub-content", ".sub_contents", "main", "article",
    ],
    "금융안심포털": [
        "#contents", "#content", ".contents", ".content",
        ".sub-content", ".sub_contents", "main", "article",
    ],
}

BREADCRUMB_SELECTORS = [
    ".breadcrumb", ".breadcrumbs", ".location", ".location-wrap",
    ".path", ".sub-location", "nav[aria-label*='breadcrumb' i]",
]


def now_kst_iso() -> str:
    return datetime.now(KST).isoformat(timespec="seconds")


def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def sha256_text(text: str) -> str:
    return sha256_bytes(text.encode("utf-8"))


def normalize_space(text: str) -> str:
    return re.sub(r"\s+", " ", text or "").strip()


def normalize_multiline(text: str) -> str:
    lines = [normalize_space(line) for line in (text or "").splitlines()]
    cleaned: list[str] = []

    for line in lines:
        if not line:
            if cleaned and cleaned[-1] != "":
                cleaned.append("")
            continue
        cleaned.append(line)

    while cleaned and cleaned[-1] == "":
        cleaned.pop()

    return "\n".join(cleaned)


def safe_name(value: str, max_length: int = 90) -> str:
    value = normalize_space(value)
    value = re.sub(r'[\\/:*?"<>|]+', "_", value)
    value = re.sub(r"_+", "_", value).strip("._ ")
    return (value or "untitled")[:max_length]


def ensure_parent(path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)


def write_json(path: Path, data: Any) -> None:
    ensure_parent(path)
    path.write_text(
        json.dumps(data, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )


def append_jsonl(path: Path, data: dict[str, Any]) -> None:
    ensure_parent(path)
    with path.open("a", encoding="utf-8") as file:
        file.write(json.dumps(data, ensure_ascii=False) + "\n")


def absolute_url(base_url: str, candidate: str | None) -> str | None:
    if not candidate:
        return None

    candidate = candidate.strip()
    if not candidate or candidate.startswith(("#", "javascript:", "mailto:", "tel:")):
        return None

    return urljoin(base_url, candidate)


def extension_from_url(url: str) -> str:
    path = urlparse(url).path
    return Path(path).suffix.lower()


def classify_asset_url(url: str) -> str:
    extension = extension_from_url(url)

    if extension in ATTACHMENT_EXTENSIONS:
        return "attachment"
    if extension in IMAGE_EXTENSIONS:
        return "image"
    if extension in VIDEO_EXTENSIONS:
        return "video"
    return "link"


def row_to_clean_dict(row: pd.Series) -> dict[str, str]:
    return {
        str(key): "" if pd.isna(value) else str(value)
        for key, value in row.to_dict().items()
    }


## 4. HTTP 세션·재시도·robots.txt 검사

In [ ]:

def create_session() -> requests.Session:
    retry_policy = Retry(
        total=MAX_RETRIES,
        connect=MAX_RETRIES,
        read=MAX_RETRIES,
        status=MAX_RETRIES,
        backoff_factor=0.8,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=frozenset({"GET", "HEAD"}),
        respect_retry_after_header=True,
        raise_on_status=False,
    )

    adapter = HTTPAdapter(
        max_retries=retry_policy,
        pool_connections=2,
        pool_maxsize=2,
    )

    session = requests.Session()
    session.headers.update(
        {
            "User-Agent": USER_AGENT,
            "Accept": (
                "text/html,application/xhtml+xml,application/xml;q=0.9,"
                "image/avif,image/webp,*/*;q=0.8"
            ),
            "Accept-Language": "ko-KR,ko;q=0.9,en;q=0.5",
            "Connection": "keep-alive",
        }
    )
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    return session


SESSION = create_session()
ROBOTS_CACHE: dict[str, RobotFileParser | None] = {}


def get_robots_parser(url: str) -> RobotFileParser | None:
    parsed = urlparse(url)
    base = f"{parsed.scheme}://{parsed.netloc}"

    if base in ROBOTS_CACHE:
        return ROBOTS_CACHE[base]

    robots_url = urljoin(base, "/robots.txt")

    try:
        response = SESSION.get(
            robots_url,
            timeout=REQUEST_TIMEOUT_SECONDS,
            allow_redirects=True,
        )

        if response.status_code >= 400:
            ROBOTS_CACHE[base] = None
            return None

        parser = RobotFileParser()
        parser.set_url(robots_url)
        parser.parse(response.text.splitlines())
        ROBOTS_CACHE[base] = parser
        return parser

    except requests.RequestException:
        ROBOTS_CACHE[base] = None
        return None


def robots_allows(url: str) -> tuple[bool, str]:
    if not RESPECT_ROBOTS_TXT:
        return True, "robots_check_disabled"

    parser = get_robots_parser(url)
    if parser is None:
        return True, "robots_unavailable_assumed_allowed"

    allowed = parser.can_fetch(USER_AGENT, url)
    return allowed, "allowed" if allowed else "disallowed"


def choose_response_encoding(response: requests.Response) -> str:
    content_type = response.headers.get("Content-Type", "")
    declared = response.encoding

    # requests의 기본 ISO-8859-1 추정은 한국어 HTML에서 잘못될 수 있습니다.
    if not declared or declared.lower() == "iso-8859-1":
        apparent = response.apparent_encoding
        if apparent:
            return apparent

    return declared or "utf-8"


## 5. HTML 수집과 응답 메타데이터 저장

In [ ]:

@dataclass
class FetchResult:
    requested_url: str
    final_url: str
    status_code: int
    ok: bool
    content_type: str
    encoding: str
    elapsed_seconds: float
    collected_at: str
    content_length: int
    raw_sha256: str
    redirect_history: list[dict[str, Any]]
    selected_headers: dict[str, str]
    body: bytes


def fetch_url(url: str) -> FetchResult:
    allowed, robots_status = robots_allows(url)
    if not allowed:
        raise PermissionError(f"robots.txt에서 수집을 허용하지 않습니다: {url}")

    started = time.perf_counter()

    response = SESSION.get(
        url,
        timeout=REQUEST_TIMEOUT_SECONDS,
        allow_redirects=True,
    )

    elapsed = time.perf_counter() - started
    encoding = choose_response_encoding(response)
    response.encoding = encoding

    selected_header_names = {
        "Content-Type",
        "Content-Length",
        "Last-Modified",
        "ETag",
        "Cache-Control",
        "Content-Disposition",
    }

    selected_headers = {
        key: value
        for key, value in response.headers.items()
        if key in selected_header_names
    }
    selected_headers["Robots-Check"] = robots_status

    return FetchResult(
        requested_url=url,
        final_url=response.url,
        status_code=response.status_code,
        ok=response.ok,
        content_type=response.headers.get("Content-Type", ""),
        encoding=encoding,
        elapsed_seconds=round(elapsed, 3),
        collected_at=now_kst_iso(),
        content_length=len(response.content),
        raw_sha256=sha256_bytes(response.content),
        redirect_history=[
            {
                "status_code": item.status_code,
                "url": item.url,
                "location": item.headers.get("Location"),
            }
            for item in response.history
        ],
        selected_headers=selected_headers,
        body=response.content,
    )


def save_fetch_result(
    manifest_row: dict[str, str],
    result: FetchResult,
) -> tuple[Path, Path]:
    business_dir = safe_name(manifest_row["business_function"])
    url_id = manifest_row["url_id"]

    html_path = PATHS["raw_html"] / business_dir / f"{url_id}.html"
    meta_path = PATHS["response_meta"] / business_dir / f"{url_id}.json"

    ensure_parent(html_path)
    html_path.write_bytes(result.body)

    metadata = asdict(result)
    metadata.pop("body", None)
    metadata["url_id"] = url_id
    metadata["business_function"] = manifest_row["business_function"]
    metadata["page_title_manifest"] = manifest_row["page_title"]
    metadata["fetch_strategy"] = manifest_row["fetch_strategy"]
    metadata["parser_profile"] = manifest_row["parser_profile"]
    metadata["raw_html_path"] = str(html_path.relative_to(OUTPUT_ROOT))

    write_json(meta_path, metadata)
    return html_path, meta_path


## 6. 결정론적 본문·표·링크·이미지 추출

In [ ]:

def element_text(tag: Tag) -> str:
    return normalize_space(tag.get_text(" ", strip=True))


def candidate_score(tag: Tag) -> float:
    text = element_text(tag)
    if not text:
        return -1.0

    link_text_length = sum(
        len(element_text(link))
        for link in tag.find_all("a")
    )
    paragraph_count = len(tag.find_all("p"))
    heading_count = len(tag.find_all(re.compile(r"^h[1-6]$")))
    table_count = len(tag.find_all("table"))

    return (
        len(text)
        - link_text_length * 0.55
        + paragraph_count * 25
        + heading_count * 35
        + table_count * 90
    )


def choose_main_root(soup: BeautifulSoup, site_name: str) -> Tag:
    selectors = MAIN_SELECTORS_BY_SITE.get(site_name, []) + ["main", "article"]

    candidates: list[Tag] = []
    seen_ids: set[int] = set()

    for selector in selectors:
        for candidate in soup.select(selector):
            if id(candidate) not in seen_ids:
                seen_ids.add(id(candidate))
                candidates.append(candidate)

    if candidates:
        return max(candidates, key=candidate_score)

    body = soup.body
    return body if isinstance(body, Tag) else soup


def clean_root(root: Tag) -> Tag:
    for selector in NOISE_SELECTORS:
        for node in root.select(selector):
            node.decompose()

    for comment in root.find_all(string=lambda text: isinstance(text, str) and not text.strip()):
        # 공백 노드는 그대로 두어도 결과에 영향이 없으므로 제거하지 않습니다.
        pass

    return root


def extract_title(
    soup: BeautifulSoup,
    root: Tag,
    manifest_title: str,
) -> str:
    for selector in ["h1", ".page-title", ".title", "h2"]:
        node = root.select_one(selector)
        if node:
            text = element_text(node)
            if 2 <= len(text) <= 180:
                return text

    if soup.title:
        text = normalize_space(soup.title.get_text(" ", strip=True))
        if text:
            return text

    return manifest_title


def extract_breadcrumb(soup: BeautifulSoup) -> list[str]:
    for selector in BREADCRUMB_SELECTORS:
        node = soup.select_one(selector)
        if not node:
            continue

        values = []
        for item in node.select("a, span, li"):
            text = element_text(item)
            if text and text not in values:
                values.append(text)

        if values:
            return values

    return []


def markdown_escape_cell(text: str) -> str:
    return normalize_space(text).replace("|", "\\|")


def table_to_markdown(table: Tag) -> tuple[str, list[list[str]]]:
    rows: list[list[str]] = []

    for tr in table.select("tr"):
        cells = [
            markdown_escape_cell(cell.get_text(" ", strip=True))
            for cell in tr.find_all(["th", "td"], recursive=False)
        ]
        if cells and any(cells):
            rows.append(cells)

    if not rows:
        return "", []

    width = max(len(row) for row in rows)
    normalized = [row + [""] * (width - len(row)) for row in rows]

    header = normalized[0]
    body = normalized[1:]

    markdown_lines = [
        "| " + " | ".join(header) + " |",
        "| " + " | ".join(["---"] * width) + " |",
    ]

    markdown_lines.extend(
        "| " + " | ".join(row) + " |"
        for row in body
    )

    return "\n".join(markdown_lines), normalized


def has_ancestor(tag: Tag, names: set[str]) -> bool:
    parent = tag.parent
    while isinstance(parent, Tag):
        if parent.name in names:
            return True
        parent = parent.parent
    return False


def root_to_markdown(root: Tag) -> tuple[str, list[dict[str, Any]]]:
    lines: list[str] = []
    blocks: list[dict[str, Any]] = []
    block_index = 0

    for node in root.select("h1,h2,h3,h4,h5,h6,p,li,table"):
        if node.name == "table":
            if has_ancestor(node, {"table"}):
                continue

            markdown, table_rows = table_to_markdown(node)
            if not markdown:
                continue

            block_index += 1
            lines.extend(["", markdown, ""])
            blocks.append(
                {
                    "block_index": block_index,
                    "block_type": "table",
                    "text": markdown,
                    "table_rows": table_rows,
                }
            )
            continue

        if has_ancestor(node, {"table"}):
            continue

        if node.name == "p" and has_ancestor(node, {"li"}):
            continue

        text = element_text(node)
        if not text:
            continue

        block_index += 1

        if re.fullmatch(r"h[1-6]", node.name or ""):
            level = int(node.name[1])
            markdown = f"{'#' * level} {text}"
            block_type = "heading"
        elif node.name == "li":
            markdown = f"- {text}"
            block_type = "list_item"
        else:
            markdown = text
            block_type = "paragraph"

        lines.extend([markdown, ""])
        blocks.append(
            {
                "block_index": block_index,
                "block_type": block_type,
                "text": text,
                "heading_level": (
                    int(node.name[1])
                    if block_type == "heading"
                    else None
                ),
            }
        )

    markdown = normalize_multiline("\n".join(lines))
    return markdown, blocks


def nearby_context(node: Tag, max_chars: int = 260) -> str:
    values: list[str] = []

    for candidate in [
        node.find_previous(["h1", "h2", "h3", "h4", "p", "li"]),
        node.parent if isinstance(node.parent, Tag) else None,
        node.find_next(["p", "li"]),
    ]:
        if isinstance(candidate, Tag):
            text = element_text(candidate)
            if text and text not in values:
                values.append(text)

    return normalize_space(" ".join(values))[:max_chars]


def extract_links(root: Tag, base_url: str) -> list[dict[str, Any]]:
    links: list[dict[str, Any]] = []
    seen: set[tuple[str, str]] = set()

    for anchor in root.find_all("a", href=True):
        url = absolute_url(base_url, anchor.get("href"))
        if not url:
            continue

        text = element_text(anchor)
        key = (url, text)
        if key in seen:
            continue
        seen.add(key)

        links.append(
            {
                "url": url,
                "anchor_text": text,
                "asset_type": classify_asset_url(url),
                "context": nearby_context(anchor),
            }
        )

    return links


def extract_images(root: Tag, base_url: str) -> list[dict[str, Any]]:
    images: list[dict[str, Any]] = []
    seen_urls: set[str] = set()

    for image in root.find_all("img"):
        candidates = [
            image.get("src"),
            image.get("data-src"),
            image.get("data-original"),
        ]

        srcset = image.get("srcset")
        if srcset:
            candidates.extend(
                item.strip().split(" ")[0]
                for item in srcset.split(",")
                if item.strip()
            )

        url = next(
            (
                absolute_url(base_url, candidate)
                for candidate in candidates
                if absolute_url(base_url, candidate)
            ),
            None,
        )

        if not url or url in seen_urls:
            continue
        seen_urls.add(url)

        caption = ""
        figure = image.find_parent("figure")
        if figure:
            figcaption = figure.find("figcaption")
            if figcaption:
                caption = element_text(figcaption)

        images.append(
            {
                "url": url,
                "alt": normalize_space(image.get("alt", "")),
                "title": normalize_space(image.get("title", "")),
                "caption": caption,
                "context": nearby_context(image),
            }
        )

    return images


def extract_videos(root: Tag, base_url: str) -> list[dict[str, Any]]:
    videos: list[dict[str, Any]] = []
    seen_urls: set[str] = set()

    for node in root.select("video[src], video source[src], iframe[src]"):
        url = absolute_url(base_url, node.get("src"))
        if not url or url in seen_urls:
            continue
        seen_urls.add(url)

        videos.append(
            {
                "url": url,
                "tag": node.name,
                "title": normalize_space(node.get("title", "")),
                "context": nearby_context(node),
            }
        )

    return videos


def parse_html_document(
    html_bytes: bytes,
    final_url: str,
    manifest_row: dict[str, str],
    encoding: str,
) -> dict[str, Any]:
    html_text = html_bytes.decode(encoding, errors="replace")
    soup = BeautifulSoup(html_text, "lxml")

    root = choose_main_root(soup, manifest_row["site_name"])
    root = clean_root(root)

    title = extract_title(
        soup=soup,
        root=root,
        manifest_title=manifest_row["page_title"],
    )
    breadcrumb = extract_breadcrumb(soup)
    markdown, blocks = root_to_markdown(root)
    links = extract_links(root, final_url)
    images = extract_images(root, final_url)
    videos = extract_videos(root, final_url)

    attachments = [
        link for link in links
        if link["asset_type"] == "attachment"
    ]

    return {
        "doc_id": manifest_row["url_id"],
        "parent_doc_id": manifest_row["url_id"],
        "title": title,
        "manifest_title": manifest_row["page_title"],
        "source_url": manifest_row["source_url"],
        "final_url": final_url,
        "site_name": manifest_row["site_name"],
        "business_function": manifest_row["business_function"],
        "target_business_function": manifest_row.get(
            "target_business_function",
            manifest_row["business_function"],
        ),
        "page_type": manifest_row["page_type"],
        "parser_profile": manifest_row["parser_profile"],
        "breadcrumb": breadcrumb,
        "content_markdown": markdown,
        "content_text": normalize_space(root.get_text(" ", strip=True)),
        "parsed_content_sha256": sha256_text(markdown),
        "blocks": blocks,
        "links": links,
        "attachments": attachments,
        "images": images,
        "videos": videos,
        "parsed_at": now_kst_iso(),
    }


## 7. 첨부파일·이미지 다운로드

In [ ]:

CONTENT_TYPE_EXTENSIONS = {
    "application/pdf": ".pdf",
    "application/zip": ".zip",
    "application/vnd.openxmlformats-officedocument.wordprocessingml.document": ".docx",
    "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet": ".xlsx",
    "application/vnd.openxmlformats-officedocument.presentationml.presentation": ".pptx",
    "image/jpeg": ".jpg",
    "image/png": ".png",
    "image/gif": ".gif",
    "image/webp": ".webp",
    "image/svg+xml": ".svg",
}


def filename_from_response(
    url: str,
    response: requests.Response,
    fallback_stem: str,
) -> str:
    disposition = response.headers.get("Content-Disposition", "")

    filename_star = re.search(
        r"filename\*=UTF-8''([^;]+)",
        disposition,
        flags=re.IGNORECASE,
    )
    filename_plain = re.search(
        r'filename="?([^";]+)"?',
        disposition,
        flags=re.IGNORECASE,
    )

    if filename_star:
        from urllib.parse import unquote
        name = unquote(filename_star.group(1))
    elif filename_plain:
        name = filename_plain.group(1)
    else:
        name = Path(urlparse(url).path).name

    name = safe_name(name, max_length=120)
    extension = Path(name).suffix.lower()

    if not extension:
        content_type = response.headers.get("Content-Type", "").split(";")[0].strip()
        extension = CONTENT_TYPE_EXTENSIONS.get(content_type)
        if not extension:
            extension = mimetypes.guess_extension(content_type) or ""
        name = f"{safe_name(fallback_stem)}{extension}"

    return name


def download_binary_asset(
    url: str,
    output_dir: Path,
    fallback_stem: str,
) -> dict[str, Any]:
    output_dir.mkdir(parents=True, exist_ok=True)

    with SESSION.get(
        url,
        timeout=REQUEST_TIMEOUT_SECONDS,
        allow_redirects=True,
        stream=True,
    ) as response:
        response.raise_for_status()

        content_length = response.headers.get("Content-Length")
        if content_length and int(content_length) > MAX_ASSET_BYTES:
            raise ValueError(
                f"파일이 제한 용량을 초과합니다: {content_length} bytes"
            )

        filename = filename_from_response(url, response, fallback_stem)
        output_path = output_dir / filename

        hasher = hashlib.sha256()
        written = 0

        with output_path.open("wb") as file:
            for chunk in response.iter_content(chunk_size=64 * 1024):
                if not chunk:
                    continue

                written += len(chunk)
                if written > MAX_ASSET_BYTES:
                    file.close()
                    output_path.unlink(missing_ok=True)
                    raise ValueError(
                        f"다운로드 중 제한 용량 {MAX_ASSET_BYTES} bytes를 초과했습니다."
                    )

                hasher.update(chunk)
                file.write(chunk)

        return {
            "source_url": url,
            "final_url": response.url,
            "filename": filename,
            "relative_path": str(output_path.relative_to(OUTPUT_ROOT)),
            "content_type": response.headers.get("Content-Type", ""),
            "size_bytes": written,
            "sha256": hasher.hexdigest(),
            "downloaded_at": now_kst_iso(),
        }


def download_document_assets(
    document: dict[str, Any],
    manifest_row: dict[str, str],
) -> dict[str, list[dict[str, Any]]]:
    business_dir = safe_name(manifest_row["business_function"])
    asset_root = (
        PATHS["raw_assets"]
        / business_dir
        / manifest_row["url_id"]
    )

    results = {
        "attachments": [],
        "images": [],
        "videos": [],
        "failures": [],
    }

    should_download_attachments = (
        DOWNLOAD_ATTACHMENTS
        and (
            manifest_row["asset_policy"] == "download_attachments"
            or manifest_row["page_type"] == "attachment_page"
        )
    )

    should_download_images = (
        DOWNLOAD_IMAGES
        and (
            manifest_row["crawl_wave"] == "W1_ASSET"
            or manifest_row["page_type"] in {
                "process_page",
                "video_page",
                "attachment_page",
            }
        )
    )

    if should_download_attachments:
        for index, item in enumerate(document["attachments"], start=1):
            try:
                result = download_binary_asset(
                    item["url"],
                    asset_root / "attachments",
                    f"{manifest_row['url_id']}_attachment_{index:03d}",
                )
                result["anchor_text"] = item.get("anchor_text", "")
                result["context"] = item.get("context", "")
                results["attachments"].append(result)
            except Exception as error:
                results["failures"].append(
                    {
                        "asset_type": "attachment",
                        "url": item["url"],
                        "error_type": type(error).__name__,
                        "error": str(error),
                    }
                )

    if should_download_images:
        for index, item in enumerate(document["images"], start=1):
            try:
                result = download_binary_asset(
                    item["url"],
                    asset_root / "images",
                    f"{manifest_row['url_id']}_image_{index:03d}",
                )
                result.update(
                    {
                        "alt": item.get("alt", ""),
                        "caption": item.get("caption", ""),
                        "context": item.get("context", ""),
                    }
                )
                results["images"].append(result)
            except Exception as error:
                results["failures"].append(
                    {
                        "asset_type": "image",
                        "url": item["url"],
                        "error_type": type(error).__name__,
                        "error": str(error),
                    }
                )

    # 영상은 기본적으로 URL·문맥 메타데이터만 보존합니다.
    if DOWNLOAD_VIDEOS:
        for index, item in enumerate(document["videos"], start=1):
            try:
                result = download_binary_asset(
                    item["url"],
                    asset_root / "videos",
                    f"{manifest_row['url_id']}_video_{index:03d}",
                )
                results["videos"].append(result)
            except Exception as error:
                results["failures"].append(
                    {
                        "asset_type": "video",
                        "url": item["url"],
                        "error_type": type(error).__name__,
                        "error": str(error),
                    }
                )

    return results


## 8. 동적 조회 페이지 진단

In [ ]:

def extract_dynamic_diagnostic(
    html_bytes: bytes,
    final_url: str,
    encoding: str,
) -> dict[str, Any]:
    html_text = html_bytes.decode(encoding, errors="replace")
    soup = BeautifulSoup(html_text, "lxml")

    forms = []
    for form_index, form in enumerate(soup.find_all("form"), start=1):
        inputs = []

        for field in form.select("input,select,textarea,button"):
            name = normalize_space(field.get("name", ""))
            field_type = normalize_space(
                field.get("type", field.name or "")
            )

            if not name and field.name != "button":
                continue

            record = {
                "tag": field.name,
                "name": name,
                "type": field_type,
                "value": normalize_space(field.get("value", "")),
            }

            if field.name == "select":
                record["options"] = [
                    {
                        "value": normalize_space(option.get("value", "")),
                        "text": element_text(option),
                    }
                    for option in field.find_all("option")
                ]

            inputs.append(record)

        forms.append(
            {
                "form_index": form_index,
                "method": normalize_space(form.get("method", "GET")).upper(),
                "action": absolute_url(final_url, form.get("action")) or final_url,
                "id": normalize_space(form.get("id", "")),
                "name": normalize_space(form.get("name", "")),
                "inputs": inputs,
            }
        )

    script_sources = []
    for script in soup.find_all("script", src=True):
        source = absolute_url(final_url, script.get("src"))
        if source and source not in script_sources:
            script_sources.append(source)

    inline_script_hints = []
    keyword_pattern = re.compile(
        r"(ajax|fetch\\s*\\(|XMLHttpRequest|\\.do\\b|pageIndex|pageNo|search)",
        flags=re.IGNORECASE,
    )

    for script in soup.find_all("script"):
        if script.get("src"):
            continue

        text = script.get_text("\\n", strip=True)
        if text and keyword_pattern.search(text):
            inline_script_hints.append(text[:1200])

    return {
        "final_url": final_url,
        "forms": forms,
        "script_sources": script_sources,
        "inline_script_hints": inline_script_hints[:20],
        "diagnosed_at": now_kst_iso(),
        "safety_note": (
            "초기 공개 페이지 구조만 진단했습니다. "
            "검색 폼 제출·로그인·본인인증·신청 실행은 수행하지 않았습니다."
        ),
    }


def ensure_playwright_installed() -> None:
    try:
        import playwright  # noqa: F401
        return
    except ImportError:
        pass

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "playwright"],
        check=True,
    )
    subprocess.run(
        [sys.executable, "-m", "playwright", "install", "chromium"],
        check=True,
    )


def save_playwright_snapshot(
    url: str,
    url_id: str,
) -> dict[str, Any] | None:
    if not ENABLE_PLAYWRIGHT_SNAPSHOT:
        return None

    ensure_playwright_installed()
    from playwright.sync_api import sync_playwright

    html_path = PATHS["dynamic_diagnostics"] / f"{url_id}_rendered.html"
    screenshot_path = PATHS["screenshots"] / f"{url_id}.png"
    ensure_parent(html_path)
    ensure_parent(screenshot_path)

    with sync_playwright() as playwright:
        browser = playwright.chromium.launch(headless=True)
        page = browser.new_page(
            user_agent=USER_AGENT,
            locale="ko-KR",
            viewport={"width": 1440, "height": 1200},
        )

        response = page.goto(
            url,
            wait_until="domcontentloaded",
            timeout=REQUEST_TIMEOUT_SECONDS * 1000,
        )
        page.wait_for_timeout(PLAYWRIGHT_WAIT_MS)

        html = page.content()
        html_path.write_text(html, encoding="utf-8")
        page.screenshot(path=str(screenshot_path), full_page=True)

        result = {
            "status_code": response.status if response else None,
            "final_url": page.url,
            "rendered_html_path": str(html_path.relative_to(OUTPUT_ROOT)),
            "screenshot_path": str(screenshot_path.relative_to(OUTPUT_ROOT)),
            "captured_at": now_kst_iso(),
        }

        browser.close()
        return result


## 9. URL 유형별 처리기

In [ ]:

def save_parsed_document(
    document: dict[str, Any],
    manifest_row: dict[str, str],
) -> tuple[Path, Path]:
    business_dir = safe_name(manifest_row["business_function"])
    url_id = manifest_row["url_id"]

    markdown_path = (
        PATHS["parsed_markdown"]
        / business_dir
        / f"{url_id}.md"
    )
    json_path = (
        PATHS["parsed_json"]
        / business_dir
        / f"{url_id}.json"
    )

    ensure_parent(markdown_path)
    markdown_path.write_text(
        document["content_markdown"],
        encoding="utf-8",
    )
    write_json(json_path, document)

    return markdown_path, json_path


def process_link_only(
    manifest_row: dict[str, str],
) -> dict[str, Any]:
    record = {
        "doc_id": manifest_row["url_id"],
        "record_type": "link_only",
        "title": manifest_row["page_title"],
        "source_url": manifest_row["source_url"],
        "site_name": manifest_row["site_name"],
        "business_function": manifest_row["business_function"],
        "target_business_function": manifest_row.get(
            "target_business_function",
            manifest_row["business_function"],
        ),
        "page_type": manifest_row["page_type"],
        "auth_required": manifest_row["auth_required"],
        "content_scope": manifest_row["content_scope"],
        "description": manifest_row.get("content_summary", ""),
        "decision_reason": manifest_row["decision_reason"],
        "created_at": now_kst_iso(),
    }

    output_path = (
        PATHS["parsed_json"]
        / safe_name(manifest_row["business_function"])
        / f"{manifest_row['url_id']}.json"
    )
    write_json(output_path, record)

    return {
        "url_id": manifest_row["url_id"],
        "business_function": manifest_row["business_function"],
        "page_title": manifest_row["page_title"],
        "source_url": manifest_row["source_url"],
        "normalized_decision": manifest_row["normalized_decision"],
        "crawl_wave": manifest_row["crawl_wave"],
        "status": "link_metadata_created",
        "status_code": None,
        "content_chars": len(record["description"]),
        "attachment_count": 0,
        "image_count": 0,
        "asset_failure_count": 0,
        "error_type": "",
        "error": "",
        "finished_at": now_kst_iso(),
    }


def process_static_or_asset(
    manifest_row: dict[str, str],
) -> dict[str, Any]:
    result = fetch_url(manifest_row["source_url"])
    html_path, meta_path = save_fetch_result(manifest_row, result)

    if not result.ok:
        raise requests.HTTPError(
            f"HTTP {result.status_code}: {result.final_url}"
        )

    if "html" not in result.content_type.lower():
        raise ValueError(
            f"HTML 응답이 아닙니다: {result.content_type}"
        )

    document = parse_html_document(
        html_bytes=result.body,
        final_url=result.final_url,
        manifest_row=manifest_row,
        encoding=result.encoding,
    )

    asset_results = download_document_assets(
        document=document,
        manifest_row=manifest_row,
    )
    document["downloaded_assets"] = asset_results

    markdown_path, json_path = save_parsed_document(
        document=document,
        manifest_row=manifest_row,
    )

    return {
        "url_id": manifest_row["url_id"],
        "business_function": manifest_row["business_function"],
        "page_title": document["title"],
        "source_url": manifest_row["source_url"],
        "final_url": result.final_url,
        "normalized_decision": manifest_row["normalized_decision"],
        "crawl_wave": manifest_row["crawl_wave"],
        "status": (
            "parse_success"
            if document["content_text"]
            else "empty_content"
        ),
        "status_code": result.status_code,
        "content_chars": len(document["content_text"]),
        "block_count": len(document["blocks"]),
        "link_count": len(document["links"]),
        "attachment_count": len(document["attachments"]),
        "image_count": len(document["images"]),
        "downloaded_attachment_count": len(asset_results["attachments"]),
        "downloaded_image_count": len(asset_results["images"]),
        "asset_failure_count": len(asset_results["failures"]),
        "raw_html_path": str(html_path.relative_to(OUTPUT_ROOT)),
        "response_meta_path": str(meta_path.relative_to(OUTPUT_ROOT)),
        "parsed_markdown_path": str(markdown_path.relative_to(OUTPUT_ROOT)),
        "parsed_json_path": str(json_path.relative_to(OUTPUT_ROOT)),
        "raw_sha256": result.raw_sha256,
        "parsed_content_sha256": document["parsed_content_sha256"],
        "error_type": "",
        "error": "",
        "finished_at": now_kst_iso(),
    }


def process_dynamic_diagnostic(
    manifest_row: dict[str, str],
) -> dict[str, Any]:
    result = fetch_url(manifest_row["source_url"])
    html_path, meta_path = save_fetch_result(manifest_row, result)

    if not result.ok:
        raise requests.HTTPError(
            f"HTTP {result.status_code}: {result.final_url}"
        )

    diagnostic = extract_dynamic_diagnostic(
        html_bytes=result.body,
        final_url=result.final_url,
        encoding=result.encoding,
    )
    diagnostic["url_id"] = manifest_row["url_id"]
    diagnostic["page_title"] = manifest_row["page_title"]
    diagnostic["business_function"] = manifest_row["business_function"]
    diagnostic["source_url"] = manifest_row["source_url"]

    playwright_snapshot = save_playwright_snapshot(
        url=manifest_row["source_url"],
        url_id=manifest_row["url_id"],
    )
    if playwright_snapshot:
        diagnostic["playwright_snapshot"] = playwright_snapshot

    diagnostic_path = (
        PATHS["dynamic_diagnostics"]
        / f"{manifest_row['url_id']}.json"
    )
    write_json(diagnostic_path, diagnostic)

    # 초기 HTML에 공개 본문이 있을 수 있으므로 파싱 결과도 참고용으로 저장합니다.
    document = parse_html_document(
        html_bytes=result.body,
        final_url=result.final_url,
        manifest_row=manifest_row,
        encoding=result.encoding,
    )
    document["collection_scope"] = "initial_public_page_only"
    markdown_path, json_path = save_parsed_document(
        document=document,
        manifest_row=manifest_row,
    )

    return {
        "url_id": manifest_row["url_id"],
        "business_function": manifest_row["business_function"],
        "page_title": document["title"],
        "source_url": manifest_row["source_url"],
        "final_url": result.final_url,
        "normalized_decision": manifest_row["normalized_decision"],
        "crawl_wave": manifest_row["crawl_wave"],
        "status": "dynamic_diagnostic_created",
        "status_code": result.status_code,
        "content_chars": len(document["content_text"]),
        "form_count": len(diagnostic["forms"]),
        "script_source_count": len(diagnostic["script_sources"]),
        "attachment_count": len(document["attachments"]),
        "image_count": len(document["images"]),
        "asset_failure_count": 0,
        "raw_html_path": str(html_path.relative_to(OUTPUT_ROOT)),
        "response_meta_path": str(meta_path.relative_to(OUTPUT_ROOT)),
        "parsed_markdown_path": str(markdown_path.relative_to(OUTPUT_ROOT)),
        "parsed_json_path": str(json_path.relative_to(OUTPUT_ROOT)),
        "dynamic_diagnostic_path": str(
            diagnostic_path.relative_to(OUTPUT_ROOT)
        ),
        "error_type": "",
        "error": "",
        "finished_at": now_kst_iso(),
    }


def process_manifest_row(
    manifest_row: dict[str, str],
) -> dict[str, Any]:
    decision = manifest_row["normalized_decision"]
    strategy = manifest_row["fetch_strategy"]

    if decision == "link_only":
        return process_link_only(manifest_row)

    if strategy == "dynamic_http_then_playwright":
        return process_dynamic_diagnostic(manifest_row)

    return process_static_or_asset(manifest_row)


## 10. 크롤링 실행

In [ ]:

CRAWL_RESULTS_JSONL = PATHS["logs"] / "crawl_results.jsonl"
CRAWL_RESULTS_CSV = PATHS["logs"] / "crawl_results.csv"
USED_MANIFEST_PATH = PATHS["manifest"] / "manifest_used.csv"

target_df.to_csv(
    USED_MANIFEST_PATH,
    index=False,
    encoding="utf-8-sig",
)

run_results: list[dict[str, Any]] = []

for _, row in tqdm(
    target_df.iterrows(),
    total=len(target_df),
    desc="크롤링 진행",
):
    manifest_row = row_to_clean_dict(row)
    started_at = now_kst_iso()

    try:
        result = process_manifest_row(manifest_row)
        result["started_at"] = started_at

    except Exception as error:
        result = {
            "url_id": manifest_row["url_id"],
            "business_function": manifest_row["business_function"],
            "page_title": manifest_row["page_title"],
            "source_url": manifest_row["source_url"],
            "normalized_decision": manifest_row["normalized_decision"],
            "crawl_wave": manifest_row["crawl_wave"],
            "status": "failed",
            "status_code": None,
            "content_chars": 0,
            "attachment_count": 0,
            "image_count": 0,
            "asset_failure_count": 0,
            "error_type": type(error).__name__,
            "error": str(error),
            "started_at": started_at,
            "finished_at": now_kst_iso(),
        }

    run_results.append(result)
    append_jsonl(CRAWL_RESULTS_JSONL, result)

    # link_only는 실제 HTTP 요청을 보내지 않으므로 대기하지 않습니다.
    if manifest_row["normalized_decision"] != "link_only":
        time.sleep(REQUEST_DELAY_SECONDS)

results_df = pd.DataFrame(run_results)
results_df.to_csv(
    CRAWL_RESULTS_CSV,
    index=False,
    encoding="utf-8-sig",
)

print("실행 완료")
display(
    results_df[
        [
            "url_id",
            "business_function",
            "status",
            "status_code",
            "content_chars",
            "attachment_count",
            "image_count",
            "asset_failure_count",
            "error_type",
        ]
    ]
)


## 11. 결과 검수

In [ ]:

if results_df.empty:
    print("실행 결과가 없습니다.")
else:
    print("상태별 건수")
    display(
        results_df.groupby("status")
        .size()
        .rename("count")
        .reset_index()
        .sort_values("count", ascending=False)
    )

    failures = results_df[results_df["status"] == "failed"].copy()
    empty_contents = results_df[
        results_df["status"].isin({"empty_content"})
        | (
            results_df["status"].isin(
                {"parse_success", "dynamic_diagnostic_created"}
            )
            & (results_df["content_chars"].fillna(0).astype(int) < 80)
        )
    ].copy()

    print("실패:", len(failures))
    print("본문 검수 필요:", len(empty_contents))

    if not failures.empty:
        display(
            failures[
                [
                    "url_id",
                    "page_title",
                    "source_url",
                    "error_type",
                    "error",
                ]
            ]
        )

    if not empty_contents.empty:
        display(
            empty_contents[
                [
                    "url_id",
                    "page_title",
                    "status",
                    "content_chars",
                ]
            ]
        )

    success_statuses = {
        "parse_success",
        "dynamic_diagnostic_created",
        "link_metadata_created",
    }
    success_count = results_df["status"].isin(success_statuses).sum()
    success_rate = (
        success_count / len(results_df) * 100
        if len(results_df)
        else 0
    )

    print(f"기본 성공률: {success_count}/{len(results_df)} ({success_rate:.1f}%)")


## 12. 페이지 문서 JSONL과 기본 청크 생성

In [ ]:

DOCUMENTS_JSONL = PATHS["processed"] / "documents.jsonl"
CHUNKS_JSONL = PATHS["processed"] / "chunks.jsonl"


def iter_parsed_json_files() -> Iterable[Path]:
    yield from sorted(PATHS["parsed_json"].rglob("*.json"))


def build_document_record(data: dict[str, Any]) -> dict[str, Any] | None:
    record_type = data.get("record_type", "page")

    if record_type == "link_only":
        return {
            "doc_id": data["doc_id"],
            "parent_doc_id": data["doc_id"],
            "record_type": "link_only",
            "title": data["title"],
            "source_url": data["source_url"],
            "site_name": data["site_name"],
            "business_function": data["business_function"],
            "target_business_function": data["target_business_function"],
            "page_type": data["page_type"],
            "content": data.get("description", ""),
            "content_hash": sha256_text(data.get("description", "")),
            "metadata": {
                "auth_required": data.get("auth_required", ""),
                "content_scope": data.get("content_scope", ""),
                "decision_reason": data.get("decision_reason", ""),
            },
        }

    content = data.get("content_markdown", "")
    if not content:
        return None

    return {
        "doc_id": data["doc_id"],
        "parent_doc_id": data.get("parent_doc_id", data["doc_id"]),
        "record_type": "page",
        "title": data.get("title", ""),
        "source_url": data.get("source_url", ""),
        "final_url": data.get("final_url", ""),
        "site_name": data.get("site_name", ""),
        "business_function": data.get("business_function", ""),
        "target_business_function": data.get(
            "target_business_function",
            data.get("business_function", ""),
        ),
        "page_type": data.get("page_type", ""),
        "breadcrumb": data.get("breadcrumb", []),
        "content": content,
        "content_hash": data.get(
            "parsed_content_sha256",
            sha256_text(content),
        ),
        "metadata": {
            "parser_profile": data.get("parser_profile", ""),
            "attachment_count": len(data.get("attachments", [])),
            "image_count": len(data.get("images", [])),
            "video_count": len(data.get("videos", [])),
            "parsed_at": data.get("parsed_at", ""),
        },
    }


def split_text_with_overlap(
    text: str,
    max_chars: int,
    overlap_chars: int,
) -> list[str]:
    text = normalize_multiline(text)
    if not text:
        return []

    paragraphs = [
        paragraph.strip()
        for paragraph in re.split(r"\n{2,}", text)
        if paragraph.strip()
    ]

    chunks: list[str] = []
    current = ""

    for paragraph in paragraphs:
        candidate = (
            f"{current}\n\n{paragraph}".strip()
            if current
            else paragraph
        )

        if len(candidate) <= max_chars:
            current = candidate
            continue

        if current:
            chunks.append(current)

        if len(paragraph) <= max_chars:
            overlap = (
                current[-overlap_chars:]
                if current and overlap_chars > 0
                else ""
            )
            current = f"{overlap}\n\n{paragraph}".strip()
            if len(current) > max_chars:
                current = paragraph
            continue

        # 매우 긴 단락은 문자 기준으로 안전하게 분할
        step = max(1, max_chars - overlap_chars)
        start = 0
        while start < len(paragraph):
            piece = paragraph[start:start + max_chars].strip()
            if piece:
                chunks.append(piece)
            start += step
        current = ""

    if current:
        chunks.append(current)

    return chunks


documents: list[dict[str, Any]] = []

for json_path in iter_parsed_json_files():
    data = json.loads(json_path.read_text(encoding="utf-8"))
    document = build_document_record(data)
    if document:
        documents.append(document)

with DOCUMENTS_JSONL.open("w", encoding="utf-8") as file:
    for document in documents:
        file.write(json.dumps(document, ensure_ascii=False) + "\n")

chunk_records: list[dict[str, Any]] = []

if CREATE_BASELINE_CHUNKS:
    for document in documents:
        for index, chunk_text in enumerate(
            split_text_with_overlap(
                document["content"],
                max_chars=CHUNK_MAX_CHARS,
                overlap_chars=CHUNK_OVERLAP_CHARS,
            )
        ):
            chunk_records.append(
                {
                    "chunk_id": f"{document['doc_id']}_chunk_{index:03d}",
                    "parent_doc_id": document["doc_id"],
                    "chunk_index": index,
                    "title": document["title"],
                    "business_function": document["business_function"],
                    "target_business_function": document[
                        "target_business_function"
                    ],
                    "page_type": document["page_type"],
                    "source_url": document["source_url"],
                    "content": chunk_text,
                    "content_hash": sha256_text(chunk_text),
                }
            )

    with CHUNKS_JSONL.open("w", encoding="utf-8") as file:
        for chunk in chunk_records:
            file.write(json.dumps(chunk, ensure_ascii=False) + "\n")

print("페이지 문서:", len(documents))
print("기본 청크:", len(chunk_records))
print("documents.jsonl:", DOCUMENTS_JSONL)
print("chunks.jsonl:", CHUNKS_JSONL)


## 13. 결과 ZIP 생성 및 다운로드

In [ ]:

archive_path = Path(
    shutil.make_archive(
        base_name=str(OUTPUT_ROOT),
        format="zip",
        root_dir=OUTPUT_ROOT.parent,
        base_dir=OUTPUT_ROOT.name,
    )
)

print("결과 ZIP:", archive_path)
print("ZIP 크기:", f"{archive_path.stat().st_size / 1024 / 1024:.2f} MB")

try:
    from google.colab import files
    files.download(str(archive_path))
except Exception as error:
    print("자동 다운로드를 시작하지 못했습니다.")
    print("Colab 왼쪽 파일 패널에서 직접 다운로드하세요:", archive_path)
    print("오류:", error)



## 결과 폴더 구조

```text
kdic_crawl_output_<실행시각>/
├── manifest/
│   └── manifest_used.csv
├── raw/
│   ├── html/
│   ├── assets/
│   └── response_meta/
├── parsed/
│   ├── markdown/
│   └── json/
├── diagnostics/
│   ├── dynamic/
│   └── screenshots/
├── processed/
│   ├── documents.jsonl
│   └── chunks.jsonl
└── logs/
    ├── crawl_results.csv
    └── crawl_results.jsonl
```

## 권장 실행 순서

1. `MAX_URLS = 3`으로 테스트
2. 실패·빈 본문·자산 오류 확인
3. 사이트별 본문 선택자와 제거 선택자 조정
4. `MAX_URLS = None`으로 전체 실행
5. `Review_Queue` 결정 후 Manifest 수정본을 업로드하여 재실행
6. 동적 조회 페이지는 진단 JSON을 검토한 뒤 공개 HTTP 요청 전용 수집기를 별도로 작성
